In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from Graphs import Node, Edge, FlowGraph

In [2]:
demand = [100, 140, 210, 180]       # units, months 1-4
prod_cost = [15, 12, 10, 14]        # €/unit
setup_cost = 200                    # € per production order
holding_cost = 1.2                  # € per unit and month in stock

def order_cost(i, j):
    """cost of producing in month i the demand of months i .. j-1"""
    months = range(i, j)
    quantity = sum(demand[m - 1] for m in months)
    storage = sum(holding_cost * demand[m - 1] * (m - i) for m in months)
    return setup_cost + prod_cost[i - 1] * quantity + storage

nodes = {i: Node(name=str(i), supply=1 if i == 1 else 0, demand=1 if i == 5 else 0) for i in range(1, 6)}
edges = [Edge(origin=nodes[i], destination=nodes[j], variable_cost=order_cost(i, j))
         for i in range(1, 5) for j in range(i + 1, 6)]

graph = FlowGraph(list(nodes.values()), edges)
graph.min_cost_flow()

Model creation time: 0.00 s
Optimal solution found: 7896.00 in 0.01 s


'Optimal'

In [3]:
pd.DataFrame(
    [(int(u), f"{u} to {int(v) - 1}", sum(demand[int(u) - 1:int(v) - 1]), d["variable_cost"])
     for u, v, d in graph.edges(data=True) if d["flow"] > 0.5],
    columns=["production month", "covers months", "units", "cost (€)"],
)

,production month,covers months,units,cost (€)
0,1,1 to 1,100,1700.0
1,2,2 to 2,140,1880.0
2,3,3 to 4,390,4316.0
